# E-Commerce Funnel & Decision Behavior Analysis

**End-to-end analysis of 20.7 million user events** from a cosmetics online store (Oct 2019 - Feb 2020).
This notebook constructs a full purchase funnel, introduces a novel **three-path purchase classification**
(Immediate / Delayed / Direct), and performs temporal, customer, and product-level analysis.

**Dataset:** [Kaggle - E-commerce Events History in Cosmetics Shop](https://www.kaggle.com/datasets/mkechinov/ecommerce-events-history-in-cosmetics-shop/data)

---


In [ ]:
# -- Import Libraries --
# Core: numpy, pandas | Viz: matplotlib, plotly | Utils: datetime, os

import numpy as np 
import pandas as pd 
from IPython.display import display, Markdown, Latex 
import plotly.express as px 
import matplotlib.pyplot as plt 
import datetime
import os 

for dirname, _, filenames in os.walk('/Users/monupaswan/Project'):
    for filename in filenames:
        print(os.path.join(dirname,filename))

## 1. Data Loading & Exploration

Load 5 monthly CSV files (Oct 2019 - Feb 2020), concatenate into a single DataFrame, and inspect the schema.


In [ ]:
# Load 5 monthly CSV files (Oct 2019 - Feb 2020)
# Oct alone has 4,102,283 rows x 9 columns

df_1910 = pd.read_csv('/Users/monupaswan/Project/2019-Oct.csv')
df_1911 = pd.read_csv('/Users/monupaswan/Project/2019-Nov.csv')
df_1912 = pd.read_csv('/Users/monupaswan/Project/2019-Dec.csv')
df_2001 = pd.read_csv('/Users/monupaswan/Project/2020-Jan.csv')
df_2002 = pd.read_csv('/Users/monupaswan/Project/2020-Feb.csv')
print(df_1910.shape)

In [ ]:
# Add month label to each DataFrame, then concatenate into one 20.7M-row DataFrame

df_1910['event_month'] = '2019-10'
df_1911['event_month'] = '2019-11'
df_1912['event_month'] = '2019-12'
df_2001['event_month'] = '2020-01'
df_2002['event_month'] = '2020-02'

df_all = pd.concat([df_1910,df_1911,df_1912,df_2001,df_2002])
df_all.head()

### Null Value Analysis

- `category_code` is **98% null** (unusable) -> use `category_id` instead
- `brand` is **42% null**
- `user_session` has only **4,598 nulls** (dropped)


In [ ]:
# Inspect the combined DataFrame - 20,692,840 rows, 10 columns, ~1.7 GB

df_all.info()

In [ ]:
# Null analysis: category_code 98% null, brand 42% null, user_session 4,598 null

df_all.isnull().sum()

In [ ]:
# Inspect rows with null user_session (these will be dropped)

nan_session = df_all[df_all['user_session'].isna()]
nan_session.sort_values(by='event_time')

In [ ]:
# Distribution of null-session events: 82% cart, 17% remove, 1% view

nan_session['event_type'].value_counts()

### Event Type Distribution

After cleaning: **20,688,242 events**. Views dominate (46.7%), cart (27.9%), remove (19.2%), purchase (6.2%).


In [ ]:
# Drop 4,598 rows with null user_session -> 20,688,242 rows remaining

df_all = df_all.dropna(subset='user_session')
df_all

In [ ]:
# Count each event type: view (9.66M), cart (5.76M), remove (3.98M), purchase (1.29M)

event_types = df_all['event_type'].value_counts()
event_types

In [ ]:
# Bar chart: Event Type Distribution

plt.figure(figsize=(12,6))
plt.title('Inspect Event Types')
plt.xlabel('Event')
plt.ylabel('Number of Events')
event_types.plot.bar()
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Convert event_time from string to datetime64[UTC] for temporal analysis

df = df_all.copy(deep=False)
df['event_time'] = pd.to_datetime(df['event_time'])

---
## 2. E-Commerce Funnel Construction

Build a **session-level pivot table** with binary flags for each event type, then compute
**nested** (sequential View->Cart->Purchase) and **non-nested** funnel metrics.


In [ ]:
# Verify datetime conversion successful

df.info()

### Nested Funnel (Sequential V -> C -> P)

Only counts sessions that followed the **strict sequence**: View first, then Cart, then Purchase.
- View->Cart: **18.4%** conversion, **81.6%** drop-off
- Cart->Purchase: **13.5%** conversion, **86.5%** drop-off
- Overall View->Purchase: **2.5%** effective rate


In [ ]:
# -- Nested Funnel Construction --
# Pivot: one row per session, binary flags for view/cart/purchase
# Then compute sequential V->VC->VCP counts and conversion/drop-off rates

sess_flags = (
    df.pivot_table(index='user_session',
                   columns='event_type',
                   values='user_id',
                   aggfunc='size',
                   fill_value=0)
)

sess_flags['has_view'] = sess_flags.get('view',0) > 0
sess_flags['has_cart'] = sess_flags.get('cart',0) > 0
sess_flags['has_purchase'] = sess_flags.get('purchase',0) > 0

sess_flags = sess_flags[['has_view','has_cart','has_purchase']]

V = sess_flags['has_view'].sum()
VC = (sess_flags['has_view'] & sess_flags['has_cart']).sum()
VCP = (sess_flags['has_view'] & sess_flags['has_cart'] & sess_flags['has_purchase']).sum()

nested_funnel = pd.DataFrame({
    'stage':['V (View)', 'VC (View+Cart)', 'VCP (View+Cart+Purchase)'],
    'sessions': [V,VC,VCP]
})
nested_funnel['next_session'] = nested_funnel['sessions'].shift(-1)
nested_funnel['Conversion_to_next_%'] = (nested_funnel['next_session']/nested_funnel['sessions'])*100
nested_funnel['drop_off_sessions'] = nested_funnel['sessions']-nested_funnel['next_session']
nested_funnel['drop_off_rate_%'] = (nested_funnel['drop_off_sessions']/nested_funnel['sessions'])*100

nested_funnel 

In [ ]:
# Bar chart: Nested funnel with conversion % and drop-off % annotations

fig, ax = plt.subplots(figsize=(8,8))

ax.bar(list(nested_funnel['stage']),nested_funnel['sessions'])

for i in range(len(nested_funnel['sessions']-1)):
    y = nested_funnel['sessions'].iloc[i]
    ax.text(i, y+13, f'{y:,}', ha='center',fontsize=12)
    ax.annotate('',xy=(i+0.9, 300000), xytext=(i+0.2, 300000), arrowprops=dict(arrowstyle='simple'))
    if(nested_funnel['drop_off_rate_%'].iloc[i] > 1):
        ax.text(i+0.5, 340000,f'-{int(nested_funnel['drop_off_rate_%'].iloc[i])}%', ha='center',fontsize=12)

ax.set_title('Shopping Funnel with Abandonment (Session-based)')
plt.tight_layout()
plt.show()

In [ ]:
# Compare non-nested (any session with event) vs nested (sequential) session counts

funnel_compare = pd.DataFrame({
    'non_nested_session' : [sess_flags['has_view'].sum(),sess_flags['has_cart'].sum(),sess_flags['has_purchase'].sum()],
    'nested_session' : [V,VC,VCP]
}, index = ['View','Cart','Purchase'])
funnel_compare

In [ ]:
# Compute the difference: sessions that cart/purchase WITHOUT a prior view event

funnel_compare['diff'] = funnel_compare['non_nested_session']-funnel_compare['nested_session']
funnel_compare

In [ ]:
# Non-nested funnel: View 4.28M -> Cart 986K (23.0%) -> Purchase 156K (15.8%)

funnel_df = pd.DataFrame({
    'stage': ['Product View', 'Add to cart', 'Purchase'],
    'sessions': [
        sess_flags['has_view'].sum(),
        sess_flags['has_cart'].sum(),
        sess_flags['has_purchase'].sum()
    ]
})

funnel_df['next_session'] = funnel_df['sessions'].shift(-1)
funnel_df['drop_off_sessions'] = funnel_df['sessions']-funnel_df['next_session']
funnel_df['drop_off_rate_%'] = (funnel_df['drop_off_sessions']/funnel_df['sessions'])*100
funnel_df['Conversion_to_next_%'] = (funnel_df['next_session']/funnel_df['sessions'])*100

funnel_df

### Non-Nested vs Nested Funnel Comparison

**Key finding:** 197,350 cart sessions and 49,091 purchase sessions occurred **without a prior view event**.
These users likely arrived via external links, saved carts, or direct product URLs.


In [ ]:
# Stacked bar: Nested (blue) vs non-nested (blue+light blue) funnel comparison

plt.figsize=(8,10)
sessions = ['View','Cart','Purchase']
fig, ax = plt.subplots(figsize=(8,8))

ax.bar(sessions, funnel_compare['nested_session'],color='lightblue')
ax.bar(sessions, funnel_compare['diff'], bottom=funnel_compare['nested_session'])

for i in range(len(nested_funnel['sessions']-1)):
    y = nested_funnel['sessions'].iloc[i]
    ax.text(i, y-70000, f'{y:,}',ha='center',fontsize=12)
    ax.annotate('',xy=(i+0.9,300000),xytext=(i+0.2,300000),arrowprops=dict(arrowstyle='simple'))
    if(nested_funnel['drop_off_rate_%'].iloc[i]>1):
        ax.text(i+0.5,340000,f'-{int(nested_funnel['drop_off_rate_%'].iloc[i])}%',ha='center',fontsize=12)

for i in range(len(funnel_df['sessions']-1)):
    y = funnel_df['sessions'].iloc[i]
    if(i>0):
        ax.text(i, y + 20000, f'{y:,}', ha='center', fontsize=12)
    ax.annotate("", xy=(i+0.9, 500000), xytext=(i+0.2, 500000),arrowprops=dict(arrowstyle="simple", color='#3776ab'))
    if(funnel_df['drop_off_rate_%'].iloc[i] > 1):
        ax.text(i+0.5, 540000, f'-{int(funnel_df['drop_off_rate_%'].iloc[i])}%', ha='center', fontsize=12, color='#3776ab')
ax.set_title("Shopping Funnel with Abandonment (Session-based)")
plt.tight_layout()
plt.show()

---
## 3. Purchase Path Classification

Classify every purchase session into one of three behavioral paths:
- **Immediate**: Session contains view, cart, AND purchase (same-session conversion)
- **Delayed**: No prior view/cart in the purchase session, but user had them in an **earlier session**
- **Direct**: No prior view/cart activity at all (possible bot / deep-link purchase)


In [ ]:
# -- Purchase Path Classification --
# Classify each purchase session as Immediate / Delayed / Direct
# Logic: merge purchase sessions with session flags, check for prior view/cart activity

purchase_session = (
    df[df['event_type']=='purchase'].groupby(['user_session','user_id'],as_index=False)
    .agg(purchase_time=('event_time','min'))
)

purchase_session = purchase_session.merge(sess_flags,left_on='user_session',right_index=True,how='left')

purchase_session['group'] = np.where(purchase_session['has_view'] & purchase_session['has_cart'] & purchase_session['has_purchase'],'Immediate','Other')

viewcart = df[df['event_type'].isin(['view','cart'])][['user_id','event_time']].copy()

first_viewcart_time = viewcart.groupby('user_id')['event_time'].min().rename('first_viewcart_time')

purchase_session = purchase_session.merge(first_viewcart_time,on='user_id',how='left')

mask_other = purchase_session['group'].eq('Other')
mask_delayed = mask_other & purchase_session['first_viewcart_time'].notna() & (purchase_session['first_viewcart_time']<purchase_session['purchase_time'])

purchase_session.loc[mask_delayed,'group'] = 'Delayed'
purchase_session.loc[mask_other & ~mask_delayed,'group'] = 'Direct'

purchase_session


In [ ]:
# Merge total revenue per purchase session into the classified DataFrame

purchase_revenue = (
    df[df['event_type']=='purchase'].groupby('user_session',as_index=False)
    .agg(revenue=('price','sum'))
)

purchase_session = purchase_session.merge(purchase_revenue, on='user_session',how='left')
purchase_session['revenue'] = purchase_session['revenue'].fillna(0)
purchase_session


In [ ]:
# Purchase session share: Immediate 68.5%, Delayed 31.0%, Direct 0.5%

session_share = purchase_session['group'].value_counts(normalize=True) * 100
session_share

In [ ]:
# Revenue share by path: nearly identical to session share

revenue_share = purchase_session.groupby('group')['revenue'].sum()
revenue_share = (revenue_share/revenue_share.sum())*100
revenue_share

### Purchase Session & Revenue Share by Path

| Path | Sessions | Share | Revenue | Share |
|------|----------|-------|---------|-------|
| Immediate | 106,526 | 68.45% | $4,341,953 | 68.40% |
| Delayed | 48,250 | 31.01% | $1,967,296 | 30.99% |
| Direct | 841 | 0.54% | $38,756 | 0.61% |


In [ ]:
# Two pie charts: session share & revenue share by path

count_of_group = purchase_session['group'].value_counts()

plt.figure(figsize=(6,6))
plt.pie(count_of_group,labels=count_of_group.index,autopct='%.1f%%',startangle=90)
plt.title('Purchase Session Share by Path')
plt.show()

rev_by_group = purchase_session.groupby('group')['revenue'].sum()

plt.figure(figsize=(6,6))
plt.pie(rev_by_group,labels=rev_by_group.index,autopct='%.1f%%',startangle=90)
plt.title('Revenue Share by Path')
plt.show()

In [ ]:
# Summary table: sessions, unique users, revenue, share % by group

summary = (
    purchase_session
    .groupby('group')
    .agg(
        purchase_session=('user_session', 'nunique'),
        unique_users=('user_id', 'nunique'),
        revenue=('revenue', 'sum')
    )
    .reset_index()
)
summary['session_share_%'] = (summary['purchase_session']/summary['purchase_session'].sum()) * 100
summary['revenue_share_%'] = (summary['revenue']/summary['revenue'].sum())*100
summary.sort_values('purchase_session',ascending=False)



In [ ]:
# Compute per-session metrics: revenue, items, purchase time for each purchase session

p = df_all[df_all['event_type']=='purchase'].copy(deep=False)

purchase_session_metrics = (
    p.groupby(['user_session','user_id'],as_index=False)
    .agg(
        revenue_session=('price','sum'),
        items_session=('product_id','count'),
        purchase_time=('event_time','min')
    )
)

purchase_session_metrics

In [ ]:
# Verify the classified purchase_session DataFrame

purchase_session

In [ ]:
# Merge group label into session metrics

purchase_session_metrics = purchase_session_metrics.merge(purchase_session[['user_session','group']],on='user_session',how='left')

purchase_session_metrics


In [ ]:
# Extended summary: AOV, avg basket size, median basket size by group

summary2 = (
    purchase_session_metrics
    .groupby('group',as_index=False)
    .agg(
        purchase_session=('user_session','nunique'),
        unique_users=('user_id','nunique'),
        revenue=('revenue_session','sum'),
        AOV=('revenue_session','mean'),
        avg_basket_size=('items_session','mean'),
        median_basket_size=('items_session','median')
    )
)
summary2['session_share_%'] = (summary2['purchase_session']/summary2['purchase_session'].sum())*100
summary2['revenue_share_%'] = (summary2['revenue']/summary2['revenue'].sum())*100

summary2.sort_values('purchase_session',ascending=False)


### Extended Metrics by Path (AOV, Basket Size, ASP)

AOV is nearly identical for Immediate ($40.76) and Delayed ($40.77), but **Direct** shows a
higher AOV ($46.08) and larger basket size (9.55 items), suggesting bulk/reseller behavior.


In [ ]:
# Average selling price per item by group (nearly uniform ~$7.8-$8.0)

purchase_session_metrics['avg_selling_price_session'] = (
    purchase_session_metrics['revenue_session']/purchase_session_metrics['items_session']
)

asp_summary = (
    purchase_session_metrics.groupby('group',as_index=False)['avg_selling_price_session']
    .mean()
)
asp_summary

In [ ]:
# Create event_date column for daily-level analysis

df['event_date'] = df['event_time'].dt.date

---
## 4. Temporal Analysis

Analyze daily, monthly, weekly, and hourly revenue patterns segmented by purchase path.


In [ ]:
# Line chart: Daily revenue trend for Immediate vs Delayed (Oct 2019 - Feb 2020)

p = df[df["event_type"] == "purchase"].copy()
p["event_time"] = pd.to_datetime(p["event_time"])
p["event_date"] = p["event_time"].dt.date

p = p.merge(purchase_session[["user_session", "group"]],
            on="user_session",
            how="left")

daily_rev_by_group = (
    p.groupby(["event_date", "group"], as_index=False)
     .agg(revenue=("price", "sum"))
)

pivot_daily = (daily_rev_by_group
               .pivot(index="event_date", columns="group", values="revenue")
               .fillna(0)
               .sort_index())

plt.figure(figsize=(12,6))
plt.plot(pivot_daily.index, pivot_daily.get("Immediate", 0), label="Immediate")
plt.plot(pivot_daily.index, pivot_daily.get("Delayed", 0), label="Delayed")

plt.title("Daily Revenue Trend: Immediate vs Delayed")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.grid(alpha=0.3)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Build monthly purchase session DataFrame with group labels

purchase_sess = (
    p.groupby(["user_session", "user_id", "event_month"], as_index=False)
     .agg(
         revenue_session=("price", "sum"),
         items_session=("product_id", "count"), 
         purchase_time=("event_time", "min")
     )
)


purchase_sess = purchase_sess.merge(
    purchase_session[["user_session", "group"]],
    on="user_session",
    how="left"
)

purchase_sess.head()

In [ ]:
# Monthly metrics: purchase sessions, revenue, AOV, avg basket size by group

monthly_metrics = (
    purchase_sess.groupby(['event_month','group'],as_index=False)
    .agg(
        purchase_session=('user_session','nunique'),
        revenue=('revenue_session','sum'),
        AOV=('revenue_session','mean'),
        avg_basket_size=('items_session','mean')
    )
)
monthly_metrics.head()



### Monthly Revenue, AOV, and Basket Size Trends

**November 2019** (Black Friday / holiday season) is the clear peak month across all groups.
Post-holiday months show significant decline, indicating strong seasonality.


In [ ]:
# Three line charts: Monthly Revenue, AOV, and Basket Size by group

def plot_metric(metric_col, title, ylabel):
    pivot = (
        monthly_metrics.pivot(index='event_month', columns='group', values=metric_col)
        .sort_index()
    )
    plt.figure(figsize=(10,5))
    for col in pivot.columns:
        plt.plot(pivot.index, pivot[col],label=col)
    plt.title(title)
    plt.xlabel('Month')
    plt.ylabel(ylabel)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_metric('revenue','Monthly Revenue by Group','Revenue')
plot_metric('AOV','Monthly AOV by Group','AOV')
plot_metric('avg_basket_size','Monthly Avg Basket Size by Group','AVG Basket Size')
    

In [ ]:
# Classify users as New (first purchase month) or Returning

first_purchase_month = (
    purchase_sess.groupby('user_id')['event_month'].min().rename('first_purchase_month')
)
purchase_sess = purchase_sess.merge(first_purchase_month, on='user_id',how='left')
purchase_sess['cust_type'] = (purchase_sess['event_month'] == purchase_sess['first_purchase_month']).map({True:'New',False:'Returning'})




---
## 5. New vs Returning Customer Analysis

Classify purchasers as **New** (first-ever purchase month) or **Returning**, then track how the mix evolves.


In [ ]:
# Verify new/returning classification

purchase_sess.head()

In [ ]:
# Monthly new vs returning unique users by group

monthly_new_return = (
    purchase_sess.groupby(['event_month','group','cust_type'],as_index=False)
    .agg(unique_users=('user_id','nunique'))
)
monthly_new_return.head()

In [ ]:
# Line chart: Monthly % new purchasing users by group

tmp = monthly_new_return.pivot_table(
    index=['event_month','group'], columns='cust_type', values='unique_users',fill_value=0
).reset_index()

tmp['new_share_%'] = (tmp['New']/(tmp['Returning']+tmp['New'])) * 100

pivot_new_share = tmp.pivot(index='event_month',columns='group',values='new_share_%').sort_index()
plt.figure(figsize=(10,5))
for col in pivot_new_share.columns:
    plt.plot(pivot_new_share.index,pivot_new_share[col],label=col)
plt.title('Monthly % New Purchasing Users by Group')
plt.xlabel('Month')
plt.ylabel('New User Share %')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()



### New User Share Trends

October starts at **~90% new buyers**. By December, the mix shifts to **~55/45 new/returning**,
demonstrating successful conversion of first-time buyers into repeat customers.


In [ ]:
# Three stacked bar charts: New vs Returning by month (one per group)

tmp_sorted = tmp.copy(deep=False)
tmp_sorted['event_month'] = tmp_sorted['event_month'].astype(str)

for g in tmp_sorted['group'].unique():
    d = tmp_sorted[tmp_sorted['group']==g].set_index('event_month')

    plt.figure(figsize=(9,4))
    plt.bar(d.index,d['New'],label='New')
    plt.bar(d.index,d['Returning'],bottom=d['New'],label='Returning')

    plt.title(f"New vs Returning Purchasing Users by Month ({g})")
    plt.xlabel("Month")
    plt.ylabel("Unique users")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Three dual-axis charts: Users (bars) + indexed Revenue/AOV/Basket (lines)

tmp2 = tmp_sorted.copy(deep=False)
tmp2['event_month'] = tmp2['event_month'].astype(str)

mm = monthly_metrics.copy(deep=False)
mm['event_month'] = mm['event_month'].astype(str)

gm = tmp2.merge(mm,on=['event_month','group'],how='left')

gm['total_users'] = gm['New']+gm['Returning']

for col in ['revenue','AOV','avg_basket_size']:
    gm[col+'_idx'] = gm.groupby('group')[col].transform(lambda s: (s/s.iloc[0])*100)

groups = gm['group'].unique()

for g in groups:
    d = gm[gm['group']==g].sort_values('event_month').copy()
    x = d['event_month'].tolist()
    
    fig, ax1 = plt.subplots(figsize=(11,4))

    ax1.bar(x,d['New'],label='New Users')
    ax1.bar(x,d['Returning'],bottom=d['New'],label='Returning Users')
    ax1.set_xlabel('Month')
    ax1.set_ylabel('Unique Users')
    ax1.grid(alpha=0.3,axis='y')

    ax2 = ax1.twinx()
    ax2.plot(x,d['AOV_idx'],marker='o',label='AOV (index)',color='red')
    ax2.plot(x,d['avg_basket_size_idx'],marker='o',label='Avg Basket Size (index)',color='black')
    ax2.plot(x,d['revenue_idx'],marker='o',label='Revenue (index)',color='g')
    ax2.set_ylabel('Metric Index (base=100)')

    ax1.set_title(f'{g}: Users (bars) vs Revenue/AOV/Basket trends(indexed lines)')

    h1,l1 = ax1.get_legend_handles_labels()
    h2,l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1+h2,l1+l2,loc='upper left',ncol=2,fontsize=9)

    plt.tight_layout()
    plt.show()


In [ ]:
# Grouped bar: New vs Returning by month across all groups

d = tmp.copy(deep=False)
d['event_month'] = d['event_month'].astype(str)

months = sorted(d['event_month'].unique())
groups = list(d['group'].unique())

x = np.arange(len(months))
width = 0.25

fig, ax = plt.subplots(figsize=(12,5))

for i,g in enumerate(groups):
    dg = d[d['group']==g].set_index('event_month').reindex(months).fillna(0)
    
    ax.bar(x+i*width,dg['New'],width,label=f'{g} - New')
    ax.bar(x+i*width,dg['Returning'],width,bottom = dg['New'],label=f'{g} - Returning')

ax.set_xticks(x+width)
ax.set_xticklabels(months,rotation=45)
ax.set_xlabel('Month')
ax.set_ylabel('Unique Users')
ax.set_title('New vs Returning Purchase Users by Month (by Group)')
ax.grid(alpha=0.3)
ax.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.show()



In [ ]:
# 100% stacked bar: New vs Returning share % by month and group

pivot_new = (d.pivot(index='event_month',columns='group',values='New').reindex(months).fillna(0))
pivot_ret = (d.pivot(index='event_month',columns='group',values='Returning').reindex(months).fillna(0))

total = pivot_new + pivot_ret
new_pct = np.where(total.values > 0, (pivot_new.values / total.values) * 100,0)
ret_pct = np.where(total.values > 0, (pivot_ret.values / total.values) * 100,0)

fig, ax = plt.subplots(figsize=(12,5))

for i, g in enumerate(groups):
    gi = list(pivot_new.columns).index(g)

    xpos = x+i*width
    new_vals = new_pct[:, gi]
    ret_vals = ret_pct[:, gi]
    
    ax.bar(xpos,new_vals,width,label=f'{g} - New')
    ax.bar(xpos,ret_vals,width,bottom=new_vals,label=f'{g} - Retruning')

    for j in range(len(months)):
        nv = new_vals[j]
        rv = ret_vals[j]

        if nv>= 3:
            ax.text(xpos[j],nv/2,f'{nv:.1f}%',ha='center',va='center',fontsize=8)
        if rv>= 3:
            ax.text(xpos[j],nv+rv/2,f'{rv:.1f}%',ha='center',va='center',fontsize=8)

ax.set_ylim(0, 100)
ax.set_ylabel("Share of users (%)")
ax.set_xlabel("Month")
ax.set_title("New vs Returning Purchasing Users by Month (100% stacked, by group)")
ax.grid(alpha=0.3, axis="y")

ax.set_xticks(x + width)
ax.set_xticklabels(months, rotation=45)

ax.legend(ncol=3, fontsize=9, bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()    


### 100% Stacked View: New vs Returning Share

This 100% stacked view makes it easy to compare the new/returning composition across groups and months.


In [ ]:
# Extract weekday and hour from purchase timestamps

p['weekday'] = p['event_time'].dt.day_name()
p['hour'] = p['event_time'].dt.hour

weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
p['weekday'] = pd.Categorical(p['weekday'],categories=weekday_order,ordered=True)
p

---
## 6. Day-of-Week & Hour-of-Day Patterns

Extract weekday and hour from purchase timestamps to identify peak purchasing windows.


In [ ]:
# Line chart: Purchase sessions by weekday (by group)

purchase_by_weekday = (
    p.groupby(['weekday','group'],as_index=False)
    .agg(purchase_session=('user_session','nunique'))
)

pivot_wd = purchase_by_weekday.pivot(index='weekday',columns='group',values='purchase_session').fillna(0).sort_index()

plt.figure(figsize=(10,5))
for col in pivot_wd.columns:
    plt.plot(pivot_wd.index.astype(str),pivot_wd[col],label=col)

plt.title("Purchase Sessions by Weekday (by group)")
plt.xlabel("Weekday")
plt.ylabel("Purchase sessions")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Three individual bar charts: Purchase sessions by weekday (one per group)

purchase_by_weekday = (
    p.groupby(['weekday','group'],observed=True,as_index=False)
    .agg(purchase_session=('user_session','nunique'))
)

pivot_wd = purchase_by_weekday.pivot(index='weekday',columns='group',values='purchase_session').fillna(0).sort_index()
weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
pivot_wd = pivot_wd.reindex(weekday_order)

for g in pivot_wd.columns:
    plt.figure(figsize=(8,6))
    plt.bar(pivot_wd.index.astype(str),pivot_wd[g])
    plt.title(f"Purchase Sessions by Weekday — {g}")
    plt.xlabel("Weekday")
    plt.ylabel("Purchase sessions")
    plt.grid(alpha=0.3, axis="y")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()



In [ ]:
# Dual-axis line: Immediate/Delayed on primary, Direct (dashed) on secondary

fig, ax1 = plt.subplots(figsize=(10,5))
for g in ['Immediate','Delayed']:
    if g in pivot_wd.columns:
        ax1.plot(pivot_wd.index.astype(str),pivot_wd[g],label=g)
ax1.set_xlabel('Weekday')
ax1.set_ylabel('Purchase Session (Immediate/Delayed)')
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
if 'Direct' in pivot_wd.columns:
    ax2.plot(pivot_wd.index.astype(str), pivot_wd['Direct'],linestyle='--',label='Direct')
ax2.set_ylabel('Purchase Session (Direct)')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2,labels1+labels2,loc='upper right')

plt.title("Purchase Sessions by Weekday (Direct on secondary axis)")
plt.tight_layout()
plt.show()



In [ ]:
# Grouped bar: Purchase sessions by weekday (all 3 groups side-by-side)

pivot_wd.plot(kind='bar',figsize=(10,5))
plt.title("Purchase Sessions by Weekday (by group)")
plt.xlabel("Weekday")
plt.ylabel("Purchase sessions")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
# Grouped bar: Revenue by weekday (all 3 groups)

revenue_by_weekday = (
    p.groupby(['weekday','group'], as_index=False)
    .agg(revenue=('price','sum'))
)
pivot_rev_wd = (revenue_by_weekday.pivot(index='weekday',columns='group',values='revenue').fillna(0).sort_index())

pivot_rev_wd.plot(kind='bar',figsize=(10,5))
plt.title("Revenue by Weekday (by group)")
plt.xlabel("Weekday")
plt.ylabel("Revenue")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



### Weekday Revenue Patterns

**Friday** is the #1 purchase day. Sunday is the lowest.
Implication: schedule promotional campaigns for Thursday evening / Friday morning.


In [ ]:
# Line chart: Purchase sessions by hour of day (0-23) by group

purchase_by_hour = p.groupby(['hour','group'],as_index=False).agg(purchase_session=('user_session','nunique'))
pivot_hr = (
    purchase_by_hour.pivot(index='hour',columns='group',values='purchase_session')
    .fillna(0)
    .sort_index()
)
plt.figure(figsize=(10,5))
for col in pivot_hr.columns:
    plt.plot(pivot_hr.index,pivot_hr[col],label=col)

plt.title("Purchase Sessions by Hour (by group)")
plt.xlabel("Hour of day")
plt.ylabel("Purchase sessions")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
# Line chart: Revenue by hour of day by group

revenue_by_hour = p.groupby(['hour','group'],as_index=False).agg(revenue=('price','sum'))
pivot_rev_hr = (
    revenue_by_hour.pivot(index='hour',columns='group',values='revenue')
    .fillna(0)
    .sort_index()
)
plt.figure(figsize=(10,5))
for col in pivot_rev_hr.columns:
    plt.plot(pivot_rev_hr.index,pivot_hr[col],label=col)

plt.title("Revenue by Hour (by group)")
plt.xlabel("Hour of day")
plt.ylabel("Revenue")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()



### Hourly Revenue Patterns

Revenue peaks between **5 PM - 8 PM** (post-work browsing).
Implication: allocate 60% of daily ad budget to the 4-9 PM window.


In [ ]:
# Dual-axis: Revenue by hour (Immediate/Delayed primary, Direct secondary)

fig, ax1 = plt.subplots(figsize=(10,5))
for g in ['Immediate','Delayed']:
    if g in pivot_rev_hr.columns:
        ax1.plot(pivot_rev_hr.index.astype(str),pivot_rev_hr[g],label=g)
ax1.set_xlabel('Hour of day')
ax1.set_ylabel('Revenue (Immediate/Delayed)')
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
if 'Direct' in pivot_wd.columns:
    ax2.plot(pivot_rev_hr.index.astype(str), pivot_rev_hr['Direct'],linestyle='--',label='Direct')
ax2.set_ylabel('Purchase Session (Direct)')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2,labels1+labels2,loc='upper right')

plt.title("Revenue by Hour (by group)") 
plt.tight_layout()
plt.show()


In [ ]:
# Top 5 categories by purchase session count, per group

top_cat_by_session = (
    p.groupby(['group','category_id'], as_index=False)
    .agg(purchase_session=('user_session','nunique'))
    .sort_values(['group','purchase_session'],ascending=[True,False])
)
top5_session = top_cat_by_session.groupby('group').head(5)
top5_session

---
## 7. Category & Product Analysis

Identify top-performing categories and products by both purchase sessions and revenue, segmented by purchase path.


In [ ]:
# Top 5 categories by revenue, per group

top_cat_by_revenue = (
    p.groupby(['group','category_id'], as_index=False)
    .agg(revenue=('price','sum'))
    .sort_values(['group','revenue'],ascending=[True,False])
)
top5_rev = top_cat_by_revenue.groupby('group').head(5)
top5_rev

In [ ]:
# Three bar charts: Top 5 categories by purchase session (one per group)

for g in p['group'].dropna().unique():
    d = top_cat_by_session[top_cat_by_session['group']==g].head(5)
    plt.figure(figsize=(8,4))
    plt.bar(d['category_id'].astype(str),d['purchase_session'])
    plt.title(f'Top 5 Category by Purchase Session ({g})')
    plt.xlabel('category_id')
    plt.ylabel('purchase_session')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()



In [ ]:
# Three bar charts: Top 5 categories by revenue (one per group)

for g in p["group"].dropna().unique():
    d = top_cat_by_revenue[top_cat_by_revenue["group"] == g].head(5)

    plt.figure(figsize=(8,4))
    plt.bar(d["category_id"].astype(str), d["revenue"])
    plt.title(f"Top 5 Category_id by Revenue({g})")
    plt.xlabel("category_id")
    plt.ylabel("revenue")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Top 5 products by purchase session (with category mapping) per group

product_cat_map = (
    p.groupby('product_id')
    .agg(
        category_id=('category_id',lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0]),
        category_code=('category_code', lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0])
    ).reset_index()
)
top_product_by_session = (
    p.groupby(['group','product_id'], as_index=False)
    .agg(purchase_session=('user_session','nunique'))
    .sort_values(['group','purchase_session'],ascending=[True,False])
)

top_product_by_session = top_product_by_session.merge(product_cat_map,on='product_id',how='left')

top5_product_session = top_product_by_session.groupby("group").head(5)
top5_product_session




### Top 5 Products by Purchase Session

Product **5809910** is the most-purchased product across all groups.
Category codes are mostly null in the dataset, so we use numeric `category_id`.


In [ ]:
# Three bar charts: Top 5 products by purchase session (one per group)

for g in p['group'].dropna().unique():
    d = top5_product_session[top5_product_session['group']==g].head(5)

    plt.figure(figsize=(8,4))
    plt.bar(d['product_id'].astype(str),d['purchase_session'])
    plt.title(f'Top 5 Category_id by Purchase Session ({g})')
    plt.xlabel("product_id")
    plt.ylabel("purchase sessions")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
# Monthly items sold pivot for top 5 products per group

purch = df_all[df_all['event_type']=='purchase'].copy(deep=False)
purch['event_time'] = pd.to_datetime(purch['event_time'])
purch['event_month'] = purch['event_time'].dt.to_period('M').astype(str)

purch = purch.merge(purchase_session[['user_session','group']],on='user_session',how='left')

top_products = top5_product_session[['group','product_id']].drop_duplicates()

purch_top = purch.merge(top_products,on=['group','product_id'],how='inner')

monthly_top_products = (
    purch_top.groupby(['group','product_id','event_month'],as_index=False)
    .agg(
        items_sold=('product_id','count'),
        purchase_session=('user_session','nunique'),
        revenue=('price','sum')
    )
)
monthly_top_products.head()




### Monthly Items Sold: Top 5 Products

Track monthly sales volume for top products. **November spike** is visible for most products.


In [ ]:
# Display pivot tables: Items sold per month for top 5 products

items_sold = ('quantity','sum')

for g in monthly_top_products['group'].unique():
    d = monthly_top_products[monthly_top_products['group']==g]

    pivot_items = (d.pivot(index='event_month',columns='product_id',values='items_sold')
                  .fillna(0)
                  .sort_index())
    print(f'\n=== {g}: Items Sold Per Month (Top 5 Products) ===')
    display(pivot_items)

pivot_rev = (d.pivot(index='event_month',columns='product_id',values='revenue')
            .fillna(0)
            .sort_index())



In [ ]:
# Three line charts: Monthly items sold for top 5 products (one per group)

for g in monthly_top_products['group'].unique():
    d = monthly_top_products[monthly_top_products['group']==g]
    pivot_items = (d.pivot(index='event_month',columns='product_id',values='items_sold')
                  .fillna(0)
                  .sort_index())
    
    plt.figure(figsize=(10,4))
    for pid in pivot_items.columns:
        plt.plot(pivot_items.index,pivot_items[pid],label=str(pid))
    plt.title(f"Monthly Items Sold — Top 5 Products ({g})")
    plt.xlabel("Month")
    plt.ylabel("Items sold")
    plt.grid(alpha=0.3)
    plt.legend(ncol=3, fontsize=8)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Top 5 products by revenue per group

top_product_by_rev = (
    p.groupby(['group','product_id'],as_index=False)
    .agg(revenue=('price','sum'))
    .sort_values(['group','revenue'],ascending=[True,False])
)

top_product_by_rev = top_product_by_rev.merge(product_cat_map, on='product_id',how='left')

top5_product_rev = top_product_by_rev.groupby('group').head(5)
top5_product_rev 


### Top 5 Products by Revenue

Product **5560754** is the #1 revenue generator despite low volume - ASP of **~$194**,
vs ~$5 for the volume leader. This volume-vs-value disconnect is a key strategic insight.


In [ ]:
# Three bar charts: Top 5 products by revenue (one per group)

for g in p['group'].unique():
    d = top5_product_rev [top5_product_rev ['group']==g]
    
    plt.figure(figsize=(8,4))
    plt.bar(d['product_id'].astype(str),d['revenue'])
    plt.title(f"Top 5 Category_id by Revenue ({g})")
    plt.xlabel("product_id")
    plt.ylabel("revenue")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    

In [ ]:
# Top 5 products by revenue with orders & items_sold count

top_product_by_rev = (
    p.groupby(["group", "product_id"], as_index=False)
     .agg(
         revenue=("price", "sum"),
         orders=("user_session", "nunique"), 
         items_sold=("product_id", "count"),      
     )
     .sort_values(["group", "revenue"], ascending=[True, False])
)

top_product_by_rev = top_product_by_rev.merge(product_cat_map, on="product_id", how="left")

top5_product_rev = top_product_by_rev.groupby("group").head(5)
top5_product_rev


### Revenue (bars) + Items Sold (lines) for Top 5 Products

The dual-axis chart reveals the **volume-value disconnect**: high-volume products do not
necessarily generate the most revenue. Promoting high-ASP products could significantly boost revenue.


In [ ]:
# Three dual-axis charts: Revenue (bars) + Items Sold (lines) for top 5 products

for g in p['group'].dropna().unique():
    d = top5_product_rev[top5_product_rev['group']==g].copy()
    d = d.sort_values('revenue',ascending=False)

    x = d['product_id'].astype(str)
    fig,ax1 = plt.subplots(figsize=(9,4))

    ax1.bar(x,d['revenue'],label='Revenue')
    ax1.set_xlabel('product_id')
    ax1.set_ylabel('Revenue')
    ax1.tick_params(axis='x',rotation=45)
    ax1.grid(alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(x,d['items_sold'],marker='',label='Items Sold',color='black')
    ax2.set_ylabel('Items Sold')

    ax1.set_title(f'Top 5 products By Revenue ({g})')

    h1,l1 = ax1.get_legend_handles_labels()
    h2,l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1+h2,l1+l2,loc='upper right')

    plt.tight_layout()
    plt.show()



---
## End of Analysis

This notebook produced **46 visualizations** covering:
- Funnel analysis (nested & non-nested)
- Three-path purchase classification
- Monthly, weekly, hourly temporal patterns
- New vs. returning customer dynamics
- Category & product deep-dives
